### LangChain chatbot with RAG - experiments

Imports and evironment

In [1]:
import random

from dotenv import load_dotenv
from langchain.agents import create_agent  # ← NOWE API!
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load environment variables from .env file with OPENAI_API_KEY
load_dotenv("../.env")

True

#### 1. RAG Documents preparation

Load document: dictionary of terms

In [2]:
documents = []

# Load dictionary PDF
dict_loader = PyPDFLoader("../documents/slownik_pojec.pdf")
dict_pages = dict_loader.load()
for page in dict_pages:
    documents.append(
        Document(
            page_content=page.page_content,
            metadata={
                "source": "slownik_pojec.pdf",
                "page": page.metadata.get("page", ""),
                "language": "pl",
                "document_type": "dictionary_of_terms",
                "publication_year": "2017",
            },
        )
    )
print(f"Loaded {len(dict_pages)} pages from slownik_pojec.pdf")

Loaded 10 pages from slownik_pojec.pdf


Load document: mortgage Loan Act

In [3]:
# Load law PDF

law_loader = PyPDFLoader("../documents/ustawa.pdf")
law_pages = law_loader.load()
for page in law_pages:
    documents.append(
        Document(
            page_content=page.page_content,
            metadata={
                "source": "ustawa.pdf",
                "page": page.metadata.get("page", ""),
                "language": "pl",
                "document_type": "law_act",
                "publication_year": "2024",
            },
        )
    )
print(f"Loaded {len(law_pages)} pages from ustawa.pdf")

Loaded 47 pages from ustawa.pdf


In [4]:
# Preview of documents

print("=== DOCUMENTS PREVIEW ===")
random_docs = random.sample(documents, min(3, len(documents)))

for i, doc in enumerate(random_docs):
    print(f"\n--- Random example {i+1} ---")
    print(f"Source: {doc.metadata['source']}")
    print(f"Page: {doc.metadata['page']}")
    print(f"Content (first 500 characters):")
    print(doc.page_content[:500] + "..." if len(doc.page_content) > 500 else doc.page_content)
    print(f"{'-'*60} \n")

=== DOCUMENTS PREVIEW ===

--- Random example 1 ---
Source: slownik_pojec.pdf
Page: 3
Content (first 500 characters):
Jakość prawna nieruchomości - Stan prawny nieruchomości określający czyste lub
obciążone prawo własności.
K
Kapitał kredytu - Kwota główna kredytu, bez odsetek i innych opłat.
Karencja w spłacie kredytu - Okres, w którym kredytobiorca nie spłaca kapitału,
płacąc tylko odsetki.
Kaucja - Kwota pieniężna składana jako zabezpieczenie wykonania zobowiązania.
Klauzula waloryzacyjna - Postanowienie umowy pozwalające na zmianę wysokości
świadczenia w zależności od zmian określonego wskaźnika.
Komornik -...
------------------------------------------------------------ 


--- Random example 2 ---
Source: ustawa.pdf
Page: 2
Content (first 500 characters):
Dziennik Ustaw – 3 – Poz. 720 
 
3) prawa własności nieruchomości gruntowej lub jej części; 
4) udziału we współwłasności budynku mieszkalnego lub lokalu mieszkalnego stanowiącego odrębną nieruchomość lub 
udziału w nieruchomości 

Chunking

In [5]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400, chunk_overlap=50, separators=["\n\n", "\n", ". ", " ", ""]
)
texts = text_splitter.split_documents(documents)

print(f"After splitting: {len(documents)} -> {len(texts)} documents\n")
print(f"Sample chunk:\n{texts[0].page_content}\n\n")

After splitting: 57 -> 592 documents

Sample chunk:
SŁOWNIK POJĘĆ
Nieruchomości i Kredyt Hipoteczny
A
Akt notarialny - Dokument sporządzony przez notariusza, potwierdzający zawarcie
umowy sprzedaży, darowizny lub innej czynności prawnej dotyczącej nieruchomości.
Aktualna wycena nieruchomości - Oszacowanie wartości rynkowej nieruchomości
przez rzeczoznawcę majątkowego, wymagane przez bank przy udzielaniu kredytu
hipotecznego.




Embeddings and vector store

In [6]:
embeddings = OpenAIEmbeddings()

# Create Chroma vector store
vectorstore = Chroma.from_documents(
    documents=texts,
    embedding=embeddings,
    collection_name="rag_vs_mortgage_loans",
)

# Check first few embeddings
print("=== SAMPLE EMBEDDING OVERVIEW ===")
sample_text = texts[0].page_content[:100]
print(f"Sample text:\n{sample_text}...\n")

# Get embedding for sample text
sample_embedding = embeddings.embed_query(sample_text)
print(f"Embedding dimension: {len(sample_embedding)}")
print(f"First 5 values: {sample_embedding[:5]}")

=== SAMPLE EMBEDDING OVERVIEW ===
Sample text:
SŁOWNIK POJĘĆ
Nieruchomości i Kredyt Hipoteczny
A
Akt notarialny - Dokument sporządzony przez notari...

Embedding dimension: 1536
First 5 values: [-0.007692780811339617, -0.0008024087874218822, 0.026530398055911064, -0.01042727380990982, -0.0037429581861943007]


#### 2. Create 'always-on RAG' Chatbot

In [ ]:
# Retriever
retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 5,
    }
)

# LLM for RAG
rag_llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

# RAG prompt
rag_template = """
You are a helpful AI assistant specialized in answering questions about mortgage loans based on provided context.
Answer the question based ONLY on the following context. 
If the answer cannot be found in the context, apologize and say you are unable to provide the answer."

Expected outputs:
a) When using the search tool:

(Precise answer based on context)
[source: filename, page: X]

b) For general questions:
(Direct answer without sources)


Context:
{context}

Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(rag_template)


# Format documents with sources
def format_docs(docs):
    formatted = []
    for doc in docs:
        source_info = f"[Source: {doc.metadata['source']}, page {doc.metadata['page']}]"
        formatted.append(f"{doc.page_content}\n{source_info}")
    return "\n\n".join(formatted)


# RAG chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()} | rag_prompt | rag_llm | StrOutputParser()
)

2.1 Testing

In [8]:
test_questions = [
    "Hi, who are you?",
    "what is mortgage?",
    "what is loan?",
    "what's the difference between mortgage and loan?",
    "tell me a joke about banks",
    "jakie są wymagania dotyczące zdolności kredytowej?",
    "co to jest hipoteka?",
    "i earned 5000 USD last month, can I get a loan?",
]

for i, question in enumerate(test_questions, 1):
    print(f"\n\n{'='*60}")
    print(f"TEST {i}/{len(test_questions)}")
    print(f"Question: {question}")
    print(f"{'-'*60}")

    result = rag_chain.invoke(question)

    print(f"Answer: {result}")



TEST 1/8
Question: Hi, who are you?
------------------------------------------------------------
Answer: I am a helpful AI assistant specialized in answering questions about mortgage loans based on the provided context. How can I assist you today?


TEST 2/8
Question: what is mortgage?
------------------------------------------------------------
Answer: Mortgage is a long-term loan secured by a mortgage on a property.  
[source: slownik_pojec.pdf, page 3]


TEST 3/8
Question: what is loan?
------------------------------------------------------------
Answer: A loan is an amount of money that is borrowed and is expected to be paid back, often with interest. Specifically, in the context of mortgage loans, it refers to the amount of money provided by a lender to a borrower, which is secured by a mortgage on a property. The borrower repays the loan in installments over time.

Based on the provided context, a mortgage loan (kredyt hipoteczny) is a long-term loan secured by a mortgage on re

________

#### 3. Create Agent with RAG tool

Tool definition

In [9]:
@tool
def search_real_estate_knowledge(question: str) -> str:
    """Retrieve relevant documents about real estate and mortgage topics."""
    try:
        # Context retrieval
        docs = retriever.invoke(question)
        
        if not docs:
            return "No relevant documents found."
            
        # Formatting
        formatted_docs = []
        for doc in docs:
            source = doc.metadata.get('source', 'unknown')
            page = doc.metadata.get('page', 'unknown')
            context = f"- {source} (page {page}): {doc.page_content}..."
            formatted_docs.append(context)

        print(f"search_real_estate_knowledge retrieved {len(formatted_docs)} documents.")
        return "\n\n".join(formatted_docs)
        
    except Exception as e:
        return f"Error retrieving documents: {str(e)}"

Creating Agent

In [ ]:
agent_llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0.0)

agent = create_agent(
    model=agent_llm,
    tools=[search_real_estate_knowledge],
    system_prompt="""

You are a helpful assistant specializing in real estate and mortgage topics.

IMPORTANT RULES:
1. When asked about real estate, mortgages, loans, or banking regulations:
   - USE the search_real_estate_knowledge tool
   - The tool returns answers WITH SOURCE REFERENCES - always include them in your response
   - Present the information clearly, keeping the source citations

2. For general conversation, jokes, or unrelated topics:
   - Answer directly WITHOUT using tools

3. General guidelines:
   - Always respond in the same language as the user's question
   - Be precise and cite sources when discussing regulations or definitions
   - If information is not in the knowledge base, say so clearly

Remember: When you use the search tool, it provides official sources. Always pass these sources to the user!

Expected outputs:
a) When using the search tool:

(Precise answer based on documents)

[Source: filename, page: X]

b) For general questions:
(Direct answer without sources)

""",
)

3.1 Testing

In [11]:
for i, question in enumerate(test_questions, 1):
    print(f"\n{'='*60}")
    print(f"TEST {i}/{len(test_questions)}")
    print(f"Question: {question}")
    print(f"{'-'*60}")

    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    print(f"Answer: {result['messages'][-1].content}")


TEST 1/8
Question: Hi, who are you?
------------------------------------------------------------
Answer: Hello! I am your helpful assistant specializing in real estate, mortgages, loans, and banking regulations. How can I assist you today?

TEST 2/8
Question: what is mortgage?
------------------------------------------------------------
search_real_estate_knowledge retrieved 5 documents.
Answer: Mortgage, or "kredyt hipoteczny" in Polish, is a long-term loan secured by a mortgage on real estate. It is a contract in which the lender provides the borrower with a loan or a promise of a loan secured by a mortgage or another right related to residential real estate. This loan is typically used to finance the purchase or maintenance of residential property not related to business activities or farming.

In essence, a mortgage is a secured loan where the property itself serves as collateral for the loan.

[Source: slownik_pojec.pdf, page 3: "Kredyt hipoteczny - Długoterminowy kredyt zabezpie